In [ ]:
from pathlib import Path

import ee
import pandas as pd

In [ ]:
# Google Earth Engine Authentication
ee.Authenticate()

In [ ]:
# Google Earth Engine Initialization
ee.Initialize(project="wetland-segmentation")

In [ ]:
# ## Set up variables
OUT_DIR = Path("data")
DIR_METADATA = OUT_DIR / "2_processed" / "0_metadata"
FILEPATH_SENTINEL2_LIST = DIR_METADATA / "sentinel2-list.csv"
SOURCE_IMAGE_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
PERCENTAGE_CLOUD_COVER = 20
TARGER_BANDS = ["B2", "B3", "B4", "B8"]
CRS = "EPSG:2958"

# Target region
aoi = ee.Geometry.Rectangle([
    -80.441228, 43.613431,  # lower left
    -80.353492, 43.674167  # upper right
])

In [ ]:
# ## Collect images
# Get the target dates
sentinel2_list = pd.read_csv(FILEPATH_SENTINEL2_LIST)
dates = sentinel2_list["date"].tolist()

# Sentinel-2 collection
collection = (
    ee.ImageCollection(SOURCE_IMAGE_COLLECTION)
    .filterBounds(aoi)
)

# Export one image per date
for date in dates:
    start = ee.Date(date)
    end = start.advance(1, "day")

    image = (
        collection
        .filterDate(start, end)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                PERCENTAGE_CLOUD_COVER
            )
        )
        .sort("CLOUDY_PIXEL_PERCENTAGE")
        .first()
    )
    if image is None:
        print(f"No image found for {date}")
        continue
    image = image.select(TARGER_BANDS)

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"S2_{date.replace('-', '')}",
        folder=OUT_DIR,
        fileNamePrefix=f"S2_{date.replace('-', '')}",
        region=aoi,
        scale=10,
        crs=CRS, # Export directly to your project CRS
        maxPixels=1e13
    )
    task.start()
    print(f"Started export: {date}")

print("All export tasks have been submitted.")